In [ ]:
# ResNet34 training on RSNA pneumonia dataset (Colab version)
#setup environemt

!pip install -q tqdm scikit-learn #for progress bar and metrics

In [ ]:
# authenticate and mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#import necessary libraries
import os
import time
import json
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
import torchvision.transforms as T

# hardware configuration, set device to gpu if available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device: ", device)

Using device:  cuda:0


In [ ]:
# define and create directory for saving training outputs (models, logs, etc.) on Google Drive
drive_dir = "/content/drive/Shareddrives/thesis/training_outputs"
os.makedirs(drive_dir, exist_ok=True)

In [ ]:
# unzip dataset from Google Drive to Colab local storage
# this speeds up the process instead of ccessing the drive one file at a time
!unzip /content/drive/Shareddrives/thesis/jpeg_dataset.zip -d /content/

Streaming output truncated to the last 5000 lines.
  inflating: /content/jpeg_dataset/train/pneumonia/3ea1ef34-b3ad-4af8-8fbd-93e46be01e24.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3ea263f3-acee-461d-bbcd-9641d9db79d9.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3ea67d3a-90be-4e77-a745-a9e323189097.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3eaa3bc8-d5dd-4104-93cd-2c217c6762f6.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3eab8ad4-5d91-4b53-a7f7-9ccc04100665.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3eb19434-abba-4221-bc02-545d97b9a093.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3eb1d6ec-9172-4b4f-be53-ca981f71e610.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3eb94a9c-7597-4c32-8ec4-37d3889fe317.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3ebe379f-90fc-44a7-9602-71c5abfdd50a.jpg  
  inflating: /content/jpeg_dataset/train/pneumonia/3ecaa81f-9128-4e65-951a-091280adb30d.jpg  
  inflati

In [ ]:
# Standard ResNet normalization and resizing
train_transforms = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transforms = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])



In [ ]:
#validate dataset
root_dir = "/content/jpeg_dataset"  

train_dataset = datasets.ImageFolder(root=f"{root_dir}/train", transform=train_transforms)
val_dataset   = datasets.ImageFolder(root=f"{root_dir}/val",   transform=val_transforms)
test_dataset  = datasets.ImageFolder(root=f"{root_dir}/test",  transform=val_transforms)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


Train samples: 11890
Val samples:   1486
Test samples:  1487


In [ ]:
# define model architecture (ResNet34) and modify the final layer for binary classification
# Using the class-based structure for better state tracking
class Resnet34Pneu(nn.Module):
    def __init__(self, pretrained=True):
        super(Resnet34Pneu, self).__init__()
        # load the pretrained ResNet34 model
        self.backbone = models.resnet34(pretrained=pretrained)
        # modify the final fully connected layer to output 2 classes (pneumonia vs normal)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, 2)
        # to track training progress within the saved file
        self.register_buffer('iter', torch.zeros(1, dtype=torch.int32))

    def forward(self, x):
        return self.backbone(x)


In [ ]:
class Trainer(object):
# this Trainer class encapsulates the training and evaluation logic, allowing for better organization and state management.
    
    def __init__(self, args): #takes a dictionary of arguments for configuration
        super(Trainer, self).__init__()
        self.args = args

    def get_loader(self, split='train'):
        # Map the split name to the actual folder path
        path = f"/content/jpeg_dataset/{split}"
        
        # to ensure the same transformations are applied when loading the dataset for training and validation/testing
        current_transform = train_transforms if split == 'train' else val_transforms   
        dataset = datasets.ImageFolder(root=path, transform=current_transform)
        
        return DataLoader(
            dataset, 
            batch_size=self.args['batch_size'], 
            shuffle=(split == 'train'), 
            num_workers=4
        )

    def train(self):
        args = self.args
        net = Resnet34Pneu().to(device) #initialize model

        # get from args, if none, use defaults
        lr = args.get('lr', 0.001) #learning rate 
        betas = args.get('betas', (0.5, 0.999)) #Adam optimizer's beta parameters for momentum and RMSprop
        weight_decay = args.get('weight_decay', 0) #regularization to prevvent overfitting

        #initialize Adam optimizer with the model parameters and hyperparameters
        optimizer = optim.Adam(net.parameters(), lr=lr, betas=betas, weight_decay=weight_decay)
        
        #loss function, it measures the distance between prediction and true label
        criterion = nn.CrossEntropyLoss()

        #check if previous model checkpoint exists, if yes, load it to resume training
        if os.path.exists(args['MODEL_DIR']):
            checkpoint = torch.load(args['MODEL_DIR'], map_location=device)
            net.load_state_dict(checkpoint['net'])
            optimizer.load_state_dict(checkpoint['optimizer'])
            print(f"Model loaded at iter: {net.iter[0].item()}")

        #prepare data loaders for training and validation
        trainloader = self.get_loader(split='train')
        valloader = self.get_loader(split='val')

        print(f'\nStart training with lr={lr}, betas={betas}...')
        start = time.time() #record start time to measure performace

        #iterator
        train_iter = iter(trainloader)
        #training loop
        for i in range(int(args['n_iter'])):
            net.train() #set model to training mode
            optimizer.zero_grad() #reset gradients

            # Pull the next batch of images (x) and labels (y0) 
            try:
                x, y0 = next(train_iter)
            except StopIteration:
                # If the dataset ends, restart the iterator (re-shuffling)
                train_iter = iter(trainloader)
                x, y0 = next(train_iter)

            # performing forward pass, compute loss, and backpropagation
            x, y0 = x.to(device), y0.to(device)
            y = net(x)
            loss = criterion(y, y0)
            loss.backward()
            optimizer.step() #adjust weights based on computed gradients

            net.iter[0] = net.iter[0] + 1

            # Track how many consecutive iterations the model stays above the target accuracy
            sustain_iter= 0
            # Every 100 iterations, check performance on validation data
            if (i + 1) % 100 == 0:
                early_stop = self.do_validate(valloader, net, VALIDATION_ACC=args['VALIDATION_ACC'])
                
                # If target accuracy is reached, add to the streak
                if early_stop:
                    sustain_iter += 100
                    print(f"Target accuracy maintained for {sustain_iter}/{args['min_iter']} iterations.")
                    
                    # If the streak hits the minimum iteration requirement, exit the loop
                    if sustain_iter >= args['min_iter']:
                        print(f'\nValidation acc consistently maintained for {args['min_iter']} iterations. Early stopping.')
                        break
                
                # If accuracy drops below the target, reset the streak entirely
                else:
                    if sustain_iter > 0:
                        print("Target accuracy lost. Resetting early stopping counter to 0.")
                    sustain_iter = 0

        end = time.time()
        print(f'\nTraining time: {round(end - start, 1)}s')

        #save into one dictionary for easier loading
        model_to_save = {'net': net.state_dict(), 'optimizer': optimizer.state_dict()}
        torch.save(model_to_save, args['MODEL_DIR'])

    # evaluate the model on validation set
    def do_validate(self, valloader, net, VALIDATION_ACC=0.8):
        net.eval() #set model to evaluation mode
        val_counter = 0 #initialize counter for correct predictions

        # disabling gradient calculation to save memory and time
        with torch.no_grad():
            for x, y0 in valloader:
                x, y0 = x.to(device), y0.to(device)
                y = net(x) #get prediction scores
                y_pred = torch.argmax(y, dim=1)
                val_counter += (y_pred == y0).sum().item()

        #calculate overall accuracy on the validation set
        acc = val_counter / len(valloader.dataset)
        print(f"Validation Acc: {acc:.4f}")
        return acc > VALIDATION_ACC

    # evaluate the model on the test set and save results
    def evaluate(self):
        #initialize test dataset
        testloader = self.get_loader(split='test')

        #load best model checkpoint
        checkpoint = torch.load(self.args['MODEL_DIR'], map_location=device)
        net = Resnet34Pneu().to(device)
        net.load_state_dict(checkpoint['net'])
        net.eval()

        #initialize counters for true positives, true negatives, false positives, and false negatives
        TP, TN, FP, FN = 0, 0, 0, 0
        with torch.no_grad():
            for x, y0 in testloader:
                x, y0 = x.to(device), y0.to(device)
                y_pred = torch.argmax(net(x), dim=1)
                for p, t in zip(y_pred, y0):
                    if p == t: #correct prediction
                        if t == 1: TP += 1 #True Positive: correctly found Pneumonia
                        else: TN += 1 #True Negative: correctly found Normal
                    else: #incorrect prediction
                        if t == 1: FN += 1 # False Negative: missed Pneumonia
                        else: FP += 1 # False Positive: healthy called Pneumonia

        results = {
            'acc': (TP + TN) / (TP + TN + FP + FN),
            'recall': TP / (TP + FN) if (TP + FN) > 0 else 0,
            'precision': TP / (TP + FP) if (TP + FP) > 0 else 0,
        }
        print('\n', results)
        #save to json file for easier analysis and reporting
        with open(self.args['RESULT_DIR'], 'w') as f:
            json.dump(results, f, indent=4)



### Training Executions


In [ ]:
# Execution 1 (resnet34_v2)
# same model configuration with Tjoa's study
args = {
    'model': 'resnet34',
    'batch_size': 32,
    'n_iter': 15000,
    'min_iter': 1000,
    'VALIDATION_ACC': 0.99,
    'MODEL_DIR': "/content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v2.pth",
    'RESULT_DIR': "/content/drive/Shareddrives/thesis/training_outputs/resultsv2.json",

    'lr': 0.001,
    'betas': (0.5, 0.999),
    'weight_decay': 0
}

trainer = Trainer(args)
trainer.train()
trainer.evaluate()

In [ ]:
# Execution 2 (resnet34_v3)
# same config, but with adjusted betas and weight decay to see if it can further improve performance
args = {
    'model': 'resnet34',
    'batch_size': 32,
    'n_iter': 15000,
    'min_iter': 1000,
    'VALIDATION_ACC': 0.99,
    'MODEL_DIR': "/content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v3.pth",
    'RESULT_DIR': "/content/drive/Shareddrives/thesis/training_outputs/resultsv3.json",

    'lr': 0.001,
    'betas': (0.9, 0.999),
    'weight_decay': 1e-5
}

trainer = Trainer(args)
trainer.train()
trainer.evaluate()

In [ ]:
# Execution 3 (resnet34_v4)
# same config as v3, but with adjusted learning rate to see if it can further improve performance
args = {
    'model': 'resnet34',
    'batch_size': 32,
    'n_iter': 15000,
    'min_iter': 1000,
    'VALIDATION_ACC': 0.99,
    'MODEL_DIR': "/content/drive/Shareddrives/thesis/checkpoints/best_resnet34_v4.pth",
    'RESULT_DIR': "/content/drive/Shareddrives/thesis/training_outputs/resultsv4.json",

    'lr': 5e-4,
    'betas': (0.9, 0.999),
    'weight_decay': 1e-5
}

trainer = Trainer(args)
trainer.train()
trainer.evaluate()

### Other Training Configurations from version 1


In [ ]:
# Training configuration #6
# batch size:32, lr=5e-4, betas=(0.9, 0.999),
# weight_decay=1e-5, target val:0.99, max iter: 15k
# with scheduler (adjusted)
# ----------------------

import torch
import torch.nn as nn
import torch.optim as optim

# Seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
import random, numpy as np
random.seed(42)
np.random.seed(42)

# Your dataset setup
root_dir = "/content/jpeg_dataset"

batch_size = 32
num_workers = 4

train_dataset = JpegRSNADataset(root_dir=root_dir, split="train", transform=train_transforms)
val_dataset   = JpegRSNADataset(root_dir=root_dir, split="val",   transform=val_transforms)
test_dataset  = JpegRSNADataset(root_dir=root_dir, split="test",  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model - Instantiating the custom Resnet34Pneu model
model = Resnet34Pneu()

model = model.to(device)

# Loss and optimizer
class_weights = torch.tensor([1.0, 1.3]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.Adam(
    model.parameters(),
    lr=5e-4,
    betas=(0.9, 0.999),
    weight_decay=1e-5
)

# --- Scheduler: ReduceLROnPlateau ---
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # minimize val_loss
    factor=0.2,        # reduce LR by 0.2
    patience=4,        # wait 4 val checks
    min_lr=1e-6
)

# Directory to save best model
model_dir = "/content/drive/Shareddrives/thesis/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "best_resnet34_6.pth")

# Iteration-based training
max_iters = 15_000
sustain_iters = 2_400
target_val_acc = 0.99
val_check_interval = 100

global_iter = 0
best_val_acc = 0.0
train_iter = iter(train_loader)
sustain_counter = 0

pbar = tqdm(total=max_iters, desc="Train", unit="iter")

if os.path.exists(model_path):
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    global_iter = checkpoint["iter"]

    print(f"Resumed training from iter {global_iter}")

# ----------------------
# Training loop
# ----------------------
while global_iter < max_iters:
    try:
        images, labels = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, labels = next(train_iter)

    model.train()
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    global_iter += 1


    pbar.update(1)

    # --- Validation & scheduler step ---
    if global_iter % val_check_interval == 0:
        val_loss, val_acc, val_recall, val_precision, val_f1 = evaluate(model, val_loader, device, criterion)

        print(
            f"Iter {global_iter}: "
            f"Val loss {val_loss:.4f}, "
            f"Val acc {val_acc:.4f}, "
            f"Val recall {val_recall:.4f}, "
            f"Val precision {val_precision:.4f}, "
            f"Val F1 {val_f1:.4f}"
        )

        # --- Step scheduler based on val_loss ---
        scheduler.step(val_loss)

        # Save log to CSV
        log_path = "/content/drive/Shareddrives/thesis/training_outputs/training_log6.csv"
        if not os.path.exists(log_path):
            with open(log_path, "w") as f:
                f.write("iter,val_loss,val_acc,val_recall,val_precision,val_f1\n")
        with open(log_path, "a") as f:
            f.write(f"{global_iter},{val_loss:.6f},{val_acc:.6f},{val_recall:.6f},{val_precision:.6f},{val_f1:.6f}\n")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iter": global_iter
            }, model_path)
            print(f"Saved new best model to {model_path}")

        # Stability-based early stopping
        if val_acc >= target_val_acc:
            sustain_counter += val_check_interval
        else:
            sustain_counter = 0

        if sustain_counter >= sustain_iters:
            print(f"Stopping early at iter {global_iter}: val_acc \u2265 {target_val_acc} sustained")
            break

pbar.close()

# Load best model for test evaluation
checkpoint = torch.load(model_path)
model.load_state_dict(checkpoint["model"])
model.eval()

test_loss, test_acc, test_recall, test_precision, test_f1 = evaluate(model, test_loader, device, criterion)
print(
    f"Test loss {test_loss:.4f}, "
    f"Test acc {test_acc:.4f}, "
    f"Test recall {test_recall:.4f}, "
    f"Test precision {test_precision:.4f}, "
    f"Test F1 {test_f1:.4f}"
)

# Save results
import json
results = {
    "best_val_acc": best_val_acc,
    "test_acc": test_acc,
    "test_recall": test_recall,
    "test_precision": test_precision,
    "test_f1": test_f1
}
results_path = "/content/drive/Shareddrives/thesis/training_outputs/results6.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=4)
print("Results saved to", results_path)


In [ ]:
# Training configuration # 1
# batch size:32, lr=1e-3, betas=(0.5, 0.999),
# weight_decay=1e-5, target val:0.99, max iter: 20k

#seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
import random
import numpy as np
random.seed(42)
np.random.seed(42)

# Point this to your dataset location in Colab
root_dir = "/content/jpeg_dataset"  # change if needed

batch_size = 32
num_workers = 4

train_dataset = JpegRSNADataset(root_dir=root_dir, split="train", transform=train_transforms)
val_dataset   = JpegRSNADataset(root_dir=root_dir, split="val",   transform=val_transforms)
test_dataset  = JpegRSNADataset(root_dir=root_dir, split="test",  transform=val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")


# Device (GPU on Colab if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load a pretrained ResNet34 backbone
model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)

# Replace the final fully connected layer for 2 classes (normal, pneumonia)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2, bias = False)

model = model.to(device)
# iteration counter
model.iter = torch.zeros(1)
print(model.fc)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    betas=(0.5, 0.999), # try also (0.9, 0.999)
    weight_decay=1e-5,
)

# Directory to save best model
model_dir = "/content/drive/Shareddrives/thesis/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "best_resnet34_1.pth")


# Iteration-based training with early stopping
max_iters = 20_000  #original is 240_000
sustain_iters = 2_400
target_val_acc = 0.99
val_check_interval = 100   # how often to run validation

global_iter = 0
best_val_acc = 0.0
train_iter = iter(train_loader)

sustain_counter = 0

pbar = tqdm(total=max_iters, desc="Train", unit="iter")

if os.path.exists(model_path):
    checkpoint = torch.load(model_path)

    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    global_iter = checkpoint["iter"]

    model.iter[0] = global_iter

    print(f"Resumed training from iter {global_iter}")

while global_iter < max_iters:
    try:
        images, labels = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, labels = next(train_iter)

    model.train()
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    model.iter[0] += 1

    global_iter += 1
    pbar.update(1)

    # Periodic validation
    if global_iter % val_check_interval == 0:
        val_loss, val_acc, val_recall, val_precision, val_f1 = evaluate(
        model, val_loader, device, criterion)

        print(
            f"Iter {global_iter}: "
            f"Val loss {val_loss:.4f}, "
            f"Val acc {val_acc:.4f}, "
            f"Val recall {val_recall:.4f}, "
            f"Val precision {val_precision:.4f}, "
            f"Val F1 {val_f1:.4f}"
        )

        # Save log to CSV in Google Drive
        log_path = "/content/drive/Shareddrives/thesis/training_outputs/training_log.csv"

        # If file doesn't exist yet, write header
        if not os.path.exists(log_path):
            with open(log_path, "w") as f:
                f.write("iter,val_loss,val_acc,val_recall,val_precision,val_f1\n")

        # Append the current validation metrics
        with open(log_path, "a") as f:
            f.write(f"{global_iter},{val_loss:.6f},{val_acc:.6f},{val_recall:.6f},{val_precision:.6f},{val_f1:.6f}\n")

        # Save best model so far
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iter": global_iter
            }, model_path)
            print(f"Saved new best model to {model_path}")
            # (keep or remove your shutil.copy to Drive as you wish)

        # --- New stability-based early stopping ---
        if val_acc >= target_val_acc:
            sustain_counter += val_check_interval  # we just passed another 'val_check_interval' iterations at/above target
        else:
            sustain_counter = 0  # reset if we dip below target

        # Stop if we've sustained target accuracy for required iterations
        if sustain_counter >= sustain_iters:
            print(
                f"Stopping early at iter {global_iter}: "
                f"val_acc maintained ≥ {target_val_acc:.2f} for {sustain_iters} iterations"
            )
            break


pbar.close()

print(f"Finished at iter {global_iter}, best val acc {best_val_acc:.4f}")

#load best model
checkpoint = torch.load(model_path)
model.load_state_dict(checkpoint["model"])
model.eval()
#test evaluation
print("\nRunning final test evaluation...")

test_loss, test_acc, test_recall, test_precision, test_f1 = evaluate(
    model, test_loader, device, criterion
)

print(
    f"Test loss {test_loss:.4f}, "
    f"Test acc {test_acc:.4f}, "
    f"Test recall {test_recall:.4f}, "
    f"Test precision {test_precision:.4f}, "
    f"Test F1 {test_f1:.4f}"
)

import json

results = {
    "best_val_acc": best_val_acc,
    "test_acc": test_acc,
    "test_recall": test_recall,
    "test_precision": test_precision,
    "test_f1": test_f1
}

results_path = "/content/drive/Shareddrives/thesis/training_outputs/results.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("Results saved to", results_path)

Train samples: 11890
Val samples:   1486
Test samples:  1487
Using device: cuda
Linear(in_features=512, out_features=2, bias=True)



Train:   0%|          | 100/20000 [00:34<1:54:07,  2.91iter/s]

Train:   0%|          | 96/20000 [00:12<34:04,  9.73iter/s]



 {'n_tested': 1486, 'loss': 0.25471185711319, 'acc': 0.8983849259757739, 'recall': 0.7687188019966722, 'precision': 0.9746835443037974, 'f1_score': 0.8595348837209301, 'TP': 462, 'TN': 873, 'FP': 12, 'FN': 139}
Iter 100: Val loss 0.2547, Val acc 0.8984, Val recall 0.7687, Val precision 0.9747, Val F1 0.8595



Train:   1%|          | 104/20000 [00:19<3:07:10,  1.77iter/s]

Saved new best model to /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_1.pth


...


Train:  99%|█████████▉| 19897/20000 [59:22<00:14,  7.07iter/s]
                                                     
Train: 100%|█████████▉| 19904/20000 [59:29<00:55,  1.73iter/s]


 {'n_tested': 1486, 'loss': 0.19413519409030155, 'acc': 0.9468371467025573, 'recall': 0.9051580698835274, 'precision': 0.9611307420494699, 'f1_score': 0.9323050556983719, 'TP': 544, 'TN': 863, 'FP': 22, 'FN': 57}
Iter 19900: Val loss 0.1941, Val acc 0.9468, Val recall 0.9052, Val precision 0.9611, Val F1 0.9323



Train: 100%|██████████| 20000/20000 [59:47<00:00,  5.58iter/s]



 {'n_tested': 1486, 'loss': 0.1921181137075655, 'acc': 0.9448183041722745, 'recall': 0.9334442595673876, 'precision': 0.9303482587064676, 'f1_score': 0.9318936877076411, 'TP': 561, 'TN': 843, 'FP': 42, 'FN': 40}
Iter 20000: Val loss 0.1921, Val acc 0.9448, Val recall 0.9334, Val precision 0.9303, Val F1 0.9319
Finished at iter 20000, best val acc 0.9563

Running final test evaluation...



 {'n_tested': 1487, 'loss': 0.1637591417896403, 'acc': 0.9394754539340955, 'recall': 0.8970099667774086, 'precision': 0.9507042253521126, 'f1_score': 0.923076923076923, 'TP': 540, 'TN': 857, 'FP': 28, 'FN': 62}
Test loss 0.1638, Test acc 0.9395, Test recall 0.8970, Test precision 0.9507, Test F1 0.9231
Results saved to /content/drive/Shareddrives/thesis/training_outputs/results.json


In [ ]:
# Training configuration # 2
# batch size:64, lr=1e-3, betas=(0.9, 0.999),
# weight_decay=1e-5, target val:0.99, max iter: 20k
# ----------------------
#seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
import random
import numpy as np
random.seed(42)
np.random.seed(42)

# Point this to your dataset location in Colab
root_dir = "/content/jpeg_dataset"  # change if needed

batch_size = 64
num_workers = 4

train_dataset = JpegRSNADataset(root_dir=root_dir, split="train", transform=train_transforms)
val_dataset   = JpegRSNADataset(root_dir=root_dir, split="val",   transform=val_transforms)
test_dataset  = JpegRSNADataset(root_dir=root_dir, split="test",  transform=val_transforms)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")


# Device (GPU on Colab if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load a pretrained ResNet34 backbone
model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)

# Replace the final fully connected layer for 2 classes (normal, pneumonia)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)

model = model.to(device)
# iteration counter
model.iter = torch.zeros(1)
print(model.fc)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999), # try also (0.9, 0.999)
    weight_decay=1e-5,
)

# Directory to save best model
model_dir = "/content/drive/Shareddrives/thesis/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "best_resnet34_2.pth")


# Iteration-based training with early stopping
max_iters = 20_000  #original is 240_000
sustain_iters = 2_400
target_val_acc = 0.99
val_check_interval = 100   # how often to run validation

global_iter = 0
best_val_acc = 0.0
train_iter = iter(train_loader)

sustain_counter = 0

pbar = tqdm(total=max_iters, desc="Train", unit="iter")

if os.path.exists(model_path):
    checkpoint = torch.load(model_path)

    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    global_iter = checkpoint["iter"]

    model.iter[0] = global_iter

    print(f"Resumed training from iter {global_iter}")

while global_iter < max_iters:
    try:
        images, labels = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, labels = next(train_iter)

    model.train()
    images = images.to(device)
    labels = labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    model.iter[0] += 1

    global_iter += 1
    pbar.update(1)

    # Periodic validation
    if global_iter % val_check_interval == 0:
        val_loss, val_acc, val_recall, val_precision, val_f1 = evaluate(
        model, val_loader, device, criterion)

        print(
            f"Iter {global_iter}: "
            f"Val loss {val_loss:.4f}, "
            f"Val acc {val_acc:.4f}, "
            f"Val recall {val_recall:.4f}, "
            f"Val precision {val_precision:.4f}, "
            f"Val F1 {val_f1:.4f}"
        )

        # Save log to CSV in Google Drive
        log_path = "/content/drive/Shareddrives/thesis/training_outputs/training_log2.csv"

        # If file doesn't exist yet, write header
        if not os.path.exists(log_path):
            with open(log_path, "w") as f:
                f.write("iter,val_loss,val_acc,val_recall,val_precision,val_f1\n")

        # Append the current validation metrics
        with open(log_path, "a") as f:
            f.write(f"{global_iter},{val_loss:.6f},{val_acc:.6f},{val_recall:.6f},{val_precision:.6f},{val_f1:.6f}\n")

        # Save best model so far
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iter": global_iter
            }, model_path)
            print(f"Saved new best model to {model_path}")
            # (keep or remove your shutil.copy to Drive as you wish)

        # --- New stability-based early stopping ---
        if val_acc >= target_val_acc:
            sustain_counter += val_check_interval  # we just passed another 'val_check_interval' iterations at/above target
        else:
            sustain_counter = 0  # reset if we dip below target

        # Stop if we've sustained target accuracy for required iterations
        if sustain_counter >= sustain_iters:
            print(
                f"Stopping early at iter {global_iter}: "
                f"val_acc maintained ≥ {target_val_acc:.2f} for {sustain_iters} iterations"
            )
            break


pbar.close()

print(f"Finished at iter {global_iter}, best val acc {best_val_acc:.4f}")

#load best model
checkpoint = torch.load(model_path)
model.load_state_dict(checkpoint["model"])
model.eval()
#test evaluation
print("\nRunning final test evaluation...")

test_loss, test_acc, test_recall, test_precision, test_f1 = evaluate(
    model, test_loader, device, criterion
)

print(
    f"Test loss {test_loss:.4f}, "
    f"Test acc {test_acc:.4f}, "
    f"Test recall {test_recall:.4f}, "
    f"Test precision {test_precision:.4f}, "
    f"Test F1 {test_f1:.4f}"
)

import json

results = {
    "best_val_acc": best_val_acc,
    "test_acc": test_acc,
    "test_recall": test_recall,
    "test_precision": test_precision,
    "test_f1": test_f1
}

results_path = "/content/drive/Shareddrives/thesis/training_outputs/results2.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("Results saved to", results_path)

Train samples: 11890
Val samples:   1486
Test samples:  1487
Using device: cuda
Linear(in_features=512, out_features=2, bias=True)


Eval:  88%|████████▊ | 21/24 [00:06<00:00,  3.90it/s]
                                                     


 {'n_tested': 1486, 'loss': 0.3912962547542269, 'acc': 0.8849259757738897, 'recall': 0.7687188019966722, 'precision': 0.9352226720647774, 'f1_score': 0.8438356164383561, 'TP': 462, 'TN': 853, 'FP': 32, 'FN': 139}
Iter 100: Val loss 0.3913, Val acc 0.8849, Val recall 0.7687, Val precision 0.9352, Val F1 0.8438


Train:   1%|          | 103/20000 [00:32<4:47:59,  1.15iter/s]

Saved new best model to /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_2.pth


...


 {'n_tested': 1486, 'loss': 0.17948566226747442, 'acc': 0.9401076716016151, 'recall': 0.8718801996672213, 'precision': 0.9776119402985075, 'f1_score': 0.9217238346525946, 'TP': 524, 'TN': 873, 'FP': 12, 'FN': 77}
Iter 1900: Val loss 0.1795, Val acc 0.9401, Val recall 0.8719, Val precision 0.9776, Val F1 0.9217


Train:  10%|█         | 2002/20000 [09:57<5:26:37,  1.09s/iter]


 {'n_tested': 1486, 'loss': 0.15770874671327154, 'acc': 0.9421265141318977, 'recall': 0.9367720465890182, 'precision': 0.9214402618657938, 'f1_score': 0.929042904290429, 'TP': 563, 'TN': 837, 'FP': 48, 'FN': 38}
Iter 2000: Val loss 0.1577, Val acc 0.9421, Val recall 0.9368, Val precision 0.9214, Val F1 0.9290


Train:  10%|█         | 2074/20000 [10:14<1:01:13,  4.88iter/s]

KeyboardInterrupt: 

In [ ]:
# Training configuration # 3
# batch size:32, lr=1e-3, betas=(0.5, 0.999),
# weight_decay=1e-5, target val:0.99, max iter: 20k
# with scheduler
# ----------------------

import torch
import torch.nn as nn
import torch.optim as optim

# Seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
import random, numpy as np
random.seed(42)
np.random.seed(42)

# Your dataset setup
root_dir = "/content/jpeg_dataset"

batch_size = 32
num_workers = 4

train_dataset = JpegRSNADataset(root_dir=root_dir, split="train", transform=train_transforms)
val_dataset   = JpegRSNADataset(root_dir=root_dir, split="val",   transform=val_transforms)
test_dataset  = JpegRSNADataset(root_dir=root_dir, split="test",  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model
model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    betas=(0.5, 0.999),
    weight_decay=1e-5
)

# --- Scheduler: ReduceLROnPlateau ---
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # minimize val_loss
    factor=0.5,        # reduce LR by 0.5
    patience=5,        # wait 5 val checks
    min_lr=1e-6
)

# Directory to save best model
model_dir = "/content/drive/Shareddrives/thesis/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "best_resnet34_3.pth")

# Iteration-based training
max_iters = 20_000
sustain_iters = 2_400
target_val_acc = 0.99
val_check_interval = 100

global_iter = 0
best_val_acc = 0.0
train_iter = iter(train_loader)
sustain_counter = 0

pbar = tqdm(total=max_iters, desc="Train", unit="iter")

if os.path.exists(model_path):
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    global_iter = checkpoint["iter"]

    print(f"Resumed training from iter {global_iter}")

# ----------------------
# Training loop
# ----------------------
while global_iter < max_iters:
    try:
        images, labels = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, labels = next(train_iter)

    model.train()
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    global_iter += 1


    pbar.update(1)

    # --- Validation & scheduler step ---
    if global_iter % val_check_interval == 0:
        val_loss, val_acc, val_recall, val_precision, val_f1 = evaluate(model, val_loader, device, criterion)

        print(
            f"Iter {global_iter}: "
            f"Val loss {val_loss:.4f}, "
            f"Val acc {val_acc:.4f}, "
            f"Val recall {val_recall:.4f}, "
            f"Val precision {val_precision:.4f}, "
            f"Val F1 {val_f1:.4f}"
        )

        # --- Step scheduler based on val_loss ---
        scheduler.step(val_loss)

        # Save log to CSV
        log_path = "/content/drive/Shareddrives/thesis/training_outputs/training_log3.csv"
        if not os.path.exists(log_path):
            with open(log_path, "w") as f:
                f.write("iter,val_loss,val_acc,val_recall,val_precision,val_f1\n")
        with open(log_path, "a") as f:
            f.write(f"{global_iter},{val_loss:.6f},{val_acc:.6f},{val_recall:.6f},{val_precision:.6f},{val_f1:.6f}\n")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iter": global_iter
            }, model_path)
            print(f"Saved new best model to {model_path}")

        # Stability-based early stopping
        if val_acc >= target_val_acc:
            sustain_counter += val_check_interval
        else:
            sustain_counter = 0

        if sustain_counter >= sustain_iters:
            print(f"Stopping early at iter {global_iter}: val_acc ≥ {target_val_acc} sustained")
            break

pbar.close()

# Load best model for test evaluation
checkpoint = torch.load(model_path)
model.load_state_dict(checkpoint["model"])
model.eval()

test_loss, test_acc, test_recall, test_precision, test_f1 = evaluate(model, test_loader, device, criterion)
print(
    f"Test loss {test_loss:.4f}, "
    f"Test acc {test_acc:.4f}, "
    f"Test recall {test_recall:.4f}, "
    f"Test precision {test_precision:.4f}, "
    f"Test F1 {test_f1:.4f}"
)

# Save results
import json
results = {
    "best_val_acc": best_val_acc,
    "test_acc": test_acc,
    "test_recall": test_recall,
    "test_precision": test_precision,
    "test_f1": test_f1
}
results_path = "/content/drive/Shareddrives/thesis/training_outputs/results3.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=4)
print("Results saved to", results_path)

Train samples: 11890
Val samples:   1486
Test samples:  1487
Using device: cuda


Eval:  98%|█████████▊| 46/47 [00:05<00:00,  8.27it/s]
                                                     


 {'n_tested': 1486, 'loss': 0.2776704513208382, 'acc': 0.9064602960969045, 'recall': 0.826955074875208, 'precision': 0.9342105263157895, 'f1_score': 0.8773168578993823, 'TP': 497, 'TN': 850, 'FP': 35, 'FN': 104}
Iter 100: Val loss 0.2777, Val acc 0.9065, Val recall 0.8270, Val precision 0.9342, Val F1 0.8773


Train:   1%|          | 105/20000 [00:19<3:14:51,  1.70iter/s]

Saved new best model to /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_3.pth


...


 {'n_tested': 1486, 'loss': 0.17394702981914298, 'acc': 0.9535666218034994, 'recall': 0.9251247920133111, 'precision': 0.9586206896551724, 'f1_score': 0.9415749364944962, 'TP': 556, 'TN': 861, 'FP': 24, 'FN': 45}
Iter 10100: Val loss 0.1739, Val acc 0.9536, Val recall 0.9251, Val precision 0.9586, Val F1 0.9416


Train:  51%|█████     | 10204/20000 [30:30<1:42:13,  1.60iter/s]


 {'n_tested': 1486, 'loss': 0.17063035654696268, 'acc': 0.9569313593539704, 'recall': 0.9351081530782029, 'precision': 0.9574105621805792, 'f1_score': 0.9461279461279462, 'TP': 562, 'TN': 860, 'FP': 25, 'FN': 39}
Iter 10200: Val loss 0.1706, Val acc 0.9569, Val recall 0.9351, Val precision 0.9574, Val F1 0.9461


Train:  51%|█████     | 10243/20000 [30:35<21:21,  7.61iter/s]

KeyboardInterrupt: 

In [ ]:
# Training configuration #4
# batch size:32, lr=1e-3, betas=(0.9, 0.999),
# weight_decay=1e-5, target val:0.99, max iter: 20k
# with scheduler (adjusted)
# ----------------------

import torch
import torch.nn as nn
import torch.optim as optim

# Seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
import random, numpy as np
random.seed(42)
np.random.seed(42)

# Your dataset setup
root_dir = "/content/jpeg_dataset"

batch_size = 32
num_workers = 4

train_dataset = JpegRSNADataset(root_dir=root_dir, split="train", transform=train_transforms)
val_dataset   = JpegRSNADataset(root_dir=root_dir, split="val",   transform=val_transforms)
test_dataset  = JpegRSNADataset(root_dir=root_dir, split="test",  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model
model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 2)
model = model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999),
    weight_decay=1e-5
)

# --- Scheduler: ReduceLROnPlateau ---
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # minimize val_loss
    factor=0.2,        # reduce LR by 0.2
    patience=4,        # wait 4 val checks
    min_lr=1e-6
)

# Directory to save best model
model_dir = "/content/drive/Shareddrives/thesis/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "best_resnet34_4.pth")

# Iteration-based training
max_iters = 20_000
sustain_iters = 2_400
target_val_acc = 0.99
val_check_interval = 100

global_iter = 0
best_val_acc = 0.0
train_iter = iter(train_loader)
sustain_counter = 0

pbar = tqdm(total=max_iters, desc="Train", unit="iter")

if os.path.exists(model_path):
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    global_iter = checkpoint["iter"]

    print(f"Resumed training from iter {global_iter}")

# ----------------------
# Training loop
# ----------------------
while global_iter < max_iters:
    try:
        images, labels = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, labels = next(train_iter)

    model.train()
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    global_iter += 1


    pbar.update(1)

    # --- Validation & scheduler step ---
    if global_iter % val_check_interval == 0:
        val_loss, val_acc, val_recall, val_precision, val_f1 = evaluate(model, val_loader, device, criterion)

        print(
            f"Iter {global_iter}: "
            f"Val loss {val_loss:.4f}, "
            f"Val acc {val_acc:.4f}, "
            f"Val recall {val_recall:.4f}, "
            f"Val precision {val_precision:.4f}, "
            f"Val F1 {val_f1:.4f}"
        )

        # --- Step scheduler based on val_loss ---
        scheduler.step(val_loss)

        # Save log to CSV
        log_path = "/content/drive/Shareddrives/thesis/training_outputs/training_log4.csv"
        if not os.path.exists(log_path):
            with open(log_path, "w") as f:
                f.write("iter,val_loss,val_acc,val_recall,val_precision,val_f1\n")
        with open(log_path, "a") as f:
            f.write(f"{global_iter},{val_loss:.6f},{val_acc:.6f},{val_recall:.6f},{val_precision:.6f},{val_f1:.6f}\n")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iter": global_iter
            }, model_path)
            print(f"Saved new best model to {model_path}")

        # Stability-based early stopping
        if val_acc >= target_val_acc:
            sustain_counter += val_check_interval
        else:
            sustain_counter = 0

        if sustain_counter >= sustain_iters:
            print(f"Stopping early at iter {global_iter}: val_acc ≥ {target_val_acc} sustained")
            break

pbar.close()

# Load best model for test evaluation
checkpoint = torch.load(model_path)
model.load_state_dict(checkpoint["model"])
model.eval()

test_loss, test_acc, test_recall, test_precision, test_f1 = evaluate(model, test_loader, device, criterion)
print(
    f"Test loss {test_loss:.4f}, "
    f"Test acc {test_acc:.4f}, "
    f"Test recall {test_recall:.4f}, "
    f"Test precision {test_precision:.4f}, "
    f"Test F1 {test_f1:.4f}"
)

# Save results
import json
results = {
    "best_val_acc": best_val_acc,
    "test_acc": test_acc,
    "test_recall": test_recall,
    "test_precision": test_precision,
    "test_f1": test_f1
}
results_path = "/content/drive/Shareddrives/thesis/training_outputs/results4.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=4)
print("Results saved to", results_path)

Train samples: 11890
Val samples:   1486
Test samples:  1487
Using device: cuda



Train:  51%|█████     | 10243/20000 [31:26<29:56,  5.43iter/s]

Train:   0%|          | 98/20000 [00:12<45:56,  7.22iter/s]



 {'n_tested': 1486, 'loss': 0.6981052781378757, 'acc': 0.7913862718707941, 'recall': 0.9534109816971714, 'precision': 0.6701754385964912, 'f1_score': 0.7870879120879122, 'TP': 573, 'TN': 603, 'FP': 282, 'FN': 28}
Iter 100: Val loss 0.6981, Val acc 0.7914, Val recall 0.9534, Val precision 0.6702, Val F1 0.7871



Train:   1%|          | 105/20000 [00:19<3:43:39,  1.48iter/s]

Saved new best model to /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_4.pth


...


Train:  99%|█████████▉| 19898/20000 [59:48<00:11,  8.80iter/s]
                                                     
Train: 100%|█████████▉| 19904/20000 [59:55<01:02,  1.53iter/s]


 {'n_tested': 1486, 'loss': 0.13387582678232507, 'acc': 0.9549125168236877, 'recall': 0.9267886855241264, 'precision': 0.9603448275862069, 'f1_score': 0.9432684165961049, 'TP': 557, 'TN': 862, 'FP': 23, 'FN': 44}
Iter 19900: Val loss 0.1339, Val acc 0.9549, Val recall 0.9268, Val precision 0.9603, Val F1 0.9433



Train: 100%|██████████| 20000/20000 [1:00:13<00:00,  5.54iter/s]



 {'n_tested': 1486, 'loss': 0.13548946047194232, 'acc': 0.9549125168236877, 'recall': 0.9267886855241264, 'precision': 0.9603448275862069, 'f1_score': 0.9432684165961049, 'TP': 557, 'TN': 862, 'FP': 23, 'FN': 44}
Iter 20000: Val loss 0.1355, Val acc 0.9549, Val recall 0.9268, Val precision 0.9603, Val F1 0.9433



 {'n_tested': 1487, 'loss': 0.14868829266068595, 'acc': 0.9408204438466712, 'recall': 0.9053156146179402, 'precision': 0.9461805555555556, 'f1_score': 0.9252971137521222, 'TP': 545, 'TN': 854, 'FP': 31, 'FN': 57}
Test loss 0.1487, Test acc 0.9408, Test recall 0.9053, Test precision 0.9462, Test F1 0.9253
Results saved to /content/drive/Shareddrives/thesis/training_outputs/results4.json


In [ ]:
# Training configuration #5 (adjust learning rate)
# batch size:32, lr=5e-4, betas=(0.9, 0.999),
# weight_decay=1e-5, target val:0.99, max iter: 15k
# with scheduler (adjusted)
# ----------------------

import torch
import torch.nn as nn
import torch.optim as optim

# Seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
import random, numpy as np
random.seed(42)
np.random.seed(42)

# Your dataset setup
root_dir = "/content/jpeg_dataset"

batch_size = 32
num_workers = 4

train_dataset = JpegRSNADataset(root_dir=root_dir, split="train", transform=train_transforms)
val_dataset   = JpegRSNADataset(root_dir=root_dir, split="val",   transform=val_transforms)
test_dataset  = JpegRSNADataset(root_dir=root_dir, split="test",  transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model - Instantiating the custom Resnet34Pneu model
model = Resnet34Pneu()

model = model.to(device)

# Loss and optimizer
class_weights = torch.tensor([1.0, 1.3]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer = optim.Adam(
    model.parameters(),
    lr=5e-4,
    betas=(0.9, 0.999),
    weight_decay=1e-5
)

# --- Scheduler: ReduceLROnPlateau ---
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # minimize val_loss
    factor=0.2,        # reduce LR by 0.2
    patience=4,        # wait 4 val checks
    min_lr=1e-6
)

# Directory to save best model
model_dir = "/content/drive/Shareddrives/thesis/checkpoints"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "best_resnet34_5.pth")

# Iteration-based training
max_iters = 15_000
sustain_iters = 2_400
target_val_acc = 0.99
val_check_interval = 100

global_iter = 0
best_val_acc = 0.0
train_iter = iter(train_loader)
sustain_counter = 0

pbar = tqdm(total=max_iters, desc="Train", unit="iter")

if os.path.exists(model_path):
    checkpoint = torch.load(model_path)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    global_iter = checkpoint["iter"]

    print(f"Resumed training from iter {global_iter}")

# ----------------------
# Training loop
# ----------------------
while global_iter < max_iters:
    try:
        images, labels = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, labels = next(train_iter)

    model.train()
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    global_iter += 1


    pbar.update(1)

    # --- Validation & scheduler step ---
    if global_iter % val_check_interval == 0:
        val_loss, val_acc, val_recall, val_precision, val_f1 = evaluate(model, val_loader, device, criterion)

        print(
            f"Iter {global_iter}: "
            f"Val loss {val_loss:.4f}, "
            f"Val acc {val_acc:.4f}, "
            f"Val recall {val_recall:.4f}, "
            f"Val precision {val_precision:.4f}, "
            f"Val F1 {val_f1:.4f}"
        )

        # --- Step scheduler based on val_loss ---
        scheduler.step(val_loss)

        # Save log to CSV
        log_path = "/content/drive/Shareddrives/thesis/training_outputs/training_log5.csv"
        if not os.path.exists(log_path):
            with open(log_path, "w") as f:
                f.write("iter,val_loss,val_acc,val_recall,val_precision,val_f1\n")
        with open(log_path, "a") as f:
            f.write(f"{global_iter},{val_loss:.6f},{val_acc:.6f},{val_recall:.6f},{val_precision:.6f},{val_f1:.6f}\n")

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "iter": global_iter
            }, model_path)
            print(f"Saved new best model to {model_path}")

        # Stability-based early stopping
        if val_acc >= target_val_acc:
            sustain_counter += val_check_interval
        else:
            sustain_counter = 0

        if sustain_counter >= sustain_iters:
            print(f"Stopping early at iter {global_iter}: val_acc ≥ {target_val_acc} sustained")
            break

pbar.close()

# Load best model for test evaluation
checkpoint = torch.load(model_path)
model.load_state_dict(checkpoint["model"])
model.eval()

test_loss, test_acc, test_recall, test_precision, test_f1 = evaluate(model, test_loader, device, criterion)
print(
    f"Test loss {test_loss:.4f}, "
    f"Test acc {test_acc:.4f}, "
    f"Test recall {test_recall:.4f}, "
    f"Test precision {test_precision:.4f}, "
    f"Test F1 {test_f1:.4f}"
)

# Save results
import json
results = {
    "best_val_acc": best_val_acc,
    "test_acc": test_acc,
    "test_recall": test_recall,
    "test_precision": test_precision,
    "test_f1": test_f1
}
results_path = "/content/drive/Shareddrives/thesis/training_outputs/results5.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=4)
print("Results saved to", results_path)


Train samples: 11890
Val samples:   1486
Test samples:  1487
Using device: cuda
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 208MB/s]
Eval: 100%|██████████| 47/47 [00:06<00:00,  8.75it/s]
                                                     


 {'n_tested': 1486, 'loss': 0.34221266783663595, 'acc': 0.9286675639300135, 'recall': 0.8585690515806988, 'precision': 0.9608938547486033, 'f1_score': 0.9068541300527241, 'TP': 516, 'TN': 864, 'FP': 21, 'FN': 85}
Iter 100: Val loss 0.3422, Val acc 0.9287, Val recall 0.8586, Val precision 0.9609, Val F1 0.9069


Train:   1%|          | 104/15000 [00:21<2:24:58,  1.71iter/s]

Saved new best model to /content/drive/Shareddrives/thesis/checkpoints/best_resnet34_5.pth


...

Train:  99%|█████████▉| 14905/15000 [44:37<00:53,  1.78iter/s]


 {'n_tested': 1486, 'loss': 0.28832619612181654, 'acc': 0.9582772543741588, 'recall': 0.9351081530782029, 'precision': 0.9606837606837607, 'f1_score': 0.9477234401349073, 'TP': 562, 'TN': 862, 'FP': 23, 'FN': 39}
Iter 14900: Val loss 0.2883, Val acc 0.9583, Val recall 0.9351, Val precision 0.9607, Val F1 0.9477


Train: 100%|██████████| 15000/15000 [44:55<00:00,  5.56iter/s]



 {'n_tested': 1486, 'loss': 0.2877899233858486, 'acc': 0.9589502018842531, 'recall': 0.940099833610649, 'precision': 0.9576271186440678, 'f1_score': 0.9487825356842988, 'TP': 565, 'TN': 860, 'FP': 25, 'FN': 36}
Iter 15000: Val loss 0.2878, Val acc 0.9590, Val recall 0.9401, Val precision 0.9576, Val F1 0.9488



 {'n_tested': 1487, 'loss': 0.2885203319022378, 'acc': 0.9562878278412912, 'recall': 0.9318936877076412, 'precision': 0.958974358974359, 'f1_score': 0.945240101095198, 'TP': 561, 'TN': 861, 'FP': 24, 'FN': 41}
Test loss 0.2885, Test acc 0.9563, Test recall 0.9319, Test precision 0.9590, Test F1 0.9452
Results saved to /content/drive/Shareddrives/thesis/training_outputs/results5.json
